# Data Space I: What the Dataset Makes Learnable

<br>

## Learning Objectives and Lesson Structure

We introduce the **data space** using a deliberately simple one-dimensional regression problem:

$$
X=\text{observed tilt angle},
\qquad
Y=\text{observed power output}.
$$

The central question is:

$$
\text{What does a finite dataset of }(x_i,y_i)\text{ points actually make learnable?}
$$

By the end, students should be able to:

- distinguish the observed dataset from the latent data-generating process;
- explain why a dataset supports claims about $P(Y\mid X)$, not automatically a physical function;
- describe how noise, missing variables, and hidden context affect what can be learned;
- identify where a finite dataset provides evidence and where a model fills gaps with assumptions;
- explain why data adequacy depends on the research claim;
- recognise when multiple plausible explanations remain compatible with the same observations;
- distinguish observed prediction, deployment prediction, and causal or physical-mechanism claims.

The notebook is organised around six questions:

1. **What can the learner see?**
2. **What claim are we trying to make?**
3. **Where does the dataset provide evidence?**
4. **Does the dataset resolve what matters?**
5. **What remains ambiguous?**
6. **Does the relationship survive a change in context?**

The recurring researcher-facing question is:

$$
\text{What claim does this dataset actually support?}
$$

<br>

### Setup

This setup cell imports the shared toy-data helpers used throughout the notebook. It changes only the Python environment for the demonstration; the data-generating examples below choose their own sample sizes, noise levels, and sampling rules. Look for the same axes, colours, and latent tilt-power curve to appear across the examples.


In [ ]:
# Environment setup. The notebook is designed to run locally and in Colab.
import os
import subprocess
import sys
import tempfile
from pathlib import Path

os.environ.setdefault(
    "MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "nextgen2026-matplotlib")
)

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

if "google.colab" in sys.modules:
    repo_dir = Path("/content/nextgen2026-mlai-workshops")
    if not repo_dir.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "workshop1",
                "https://github.com/nextgenerationgraduatesprogram/nextgen2026-mlai-workshops.git",
                str(repo_dir),
            ],
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(repo_dir)], check=True)
    sys.path.insert(0, str(repo_dir / "src"))
else:
    for possible_root in (Path.cwd(), Path.cwd().parent):
        possible_src = possible_root / "src"
        if (possible_src / "nextgen2026_mlai_workshops").exists():
            sys.path.insert(0, str(possible_src))
            break

In [ ]:
import sys

import matplotlib.pyplot as plt
import numpy as np

module_name = "nextgen2026_mlai_workshops"
for loaded_name in list(sys.modules):
    if loaded_name == module_name or loaded_name.startswith(f"{module_name}."):
        del sys.modules[loaded_name]

from nextgen2026_mlai_workshops.data_space import (
    configure_matplotlib,
    plot_ambiguity_example,
    plot_claim_targets_example,
    plot_context_shift_example,
    plot_local_support_example,
    plot_observability_example,
    plot_resolution_diagnostic_example,
)

configure_matplotlib()

print("Setup complete. Change the variables at the top of each example cell and rerun it.")


<br>

## 1. What Can the Learner See?

### Motivation

A dataset contains measurements, not the full data-generating process. The learner receives the observed sample:

$$
\mathcal{D}_{\mathrm{obs}}=\{(x_i,y_i)\}_{i=1}^{n}.
$$

In the toy problem, $X$ is measured tilt and $Y$ is measured output. A simple way to describe the process behind those observations is:

$$
X=\Theta+\varepsilon_X,
$$

$$
Y=f(\Theta,C)+\varepsilon_Y.
$$

Here $\Theta$ is true physical tilt, $C$ is hidden context, and $\varepsilon_X,\varepsilon_Y$ are measurement and output noise. The full latent record would contain $(\Theta,C,X,Y)$, but the learner only sees $(X,Y)$.

Before fitting a model, we need to separate the observable variables from the variables used in the researcher's explanation. The most direct target available from representative observed samples is:

$$
m_{\mathrm{obs}}(x)=\mathbb{E}_{P_{\mathrm{obs}}}[Y\mid X=x].
$$

This section asks:

$$
\text{What information is actually available to the learner?}
$$

### Minimal example

The plot shows the same dataset in two ways.

In the learner view, each point is just an observed pair $(X,Y)$. In the teaching view, we reveal hidden context $C$ to show one possible reason why similar tilt values can produce different outputs.

The variables at the top of the cell control the observation process: sample size $n$, measurement noise in $X$, output noise in $Y$, hidden-context strength, and the sampling rule for $\Theta$.

Look for the difference between:

- the variables available to the learner;
- the hidden variables used to explain the data;
- the relationship a model can learn from $(X,Y)$ alone.


In [ ]:
# Change these values, then rerun this cell.
# sampling options: "uniform", "cluster40", "restricted", "gap65", "sparse_feature", "dense_feature"
n = 170
seed = 21
x_noise = 4.0
y_noise = 0.06
context_strength = 0.35
sampling = "uniform"

result = plot_observability_example(
    n=n,
    seed=seed,
    x_noise=x_noise,
    y_noise=y_noise,
    context_strength=context_strength,
    sampling=sampling,
)


### Plot interpretation

The learner view supports claims about the observed relationship between measured tilt and measured output.

The teaching view shows that this observed relationship may be a mixture of different hidden contexts. Once $C$ is revealed, some variation in $Y$ is no longer just unexplained noise. It reflects structure that was missing from the learner's dataset.

It should be read as:

$$
\text{the scatter plot reveals what is observable from the measured variables.}
$$

If $C$ is not measured, a model trained on $(X,Y)$ cannot condition on $C$. It must learn from the observed mixture. From representative observed samples, that mixture defines $m_{\mathrm{obs}}(x)$, not automatically the clean physical curve $f_0$.

### Takeaway

The first limit of a dataset is **observability**.

A model trained on $(X,Y)$ can learn relationships expressed in the observed variables. It cannot separate hidden contexts, correct measurement error, or recover the physical mechanism unless the needed variables or assumptions are also available.

The practical question is:

$$
\text{What exactly did the dataset measure?}
$$


<br>

## 2. What Claim Are We Trying to Make?

### Motivation

The same dataset can support different kinds of claims. A model trained on observed $(X,Y)$ may be useful for prediction, but that does not mean it has recovered the physical mechanism that generated the data.

Formally, a research claim chooses an estimand: the relationship or functional we want the data to identify. Different estimands can be attached to the same observed scatter plot.

For observed prediction, the natural target is:

$$
m_{\mathrm{obs}}(x)=\mathbb{E}_{P_{\mathrm{obs}}}[Y\mid X=x].
$$

For a physical-response claim, the target might instead be the latent curve:

$$
f_0(\theta).
$$

For deployment prediction, the target changes with the deployment distribution:

$$
m_{\mathrm{deploy}}(x)=\mathbb{E}_{P_{\mathrm{deploy}}}[Y\mid X=x].
$$

For an intervention claim, the target is different again:

$$
\mathbb{E}[Y\mid do(\Theta=x)].
$$

This section asks:

$$
\text{What relationship are we actually trying to estimate?}
$$

That choice matters because each estimand requires different evidence.

### Claim checklist

| Research claim | Target | What the data need to support |
|---|---|---|
| Predict observed output from observed tilt | $m_{\mathrm{obs}}(x)=\mathbb{E}_{P_{\mathrm{obs}}}[Y\mid X=x]$ | Representative observed $(X,Y)$ samples |
| Recover the physical response | $f_0(\theta)$ | Access to true tilt $\Theta$ or justified assumptions linking $X$ to $\Theta$ |
| Predict deployment behaviour | $m_{\mathrm{deploy}}(x)=\mathbb{E}_{P_{\mathrm{deploy}}}[Y\mid X=x]$ | Data covering deployment conditions |
| Make a causal claim | $\mathbb{E}[Y\mid do(\Theta=x)]$ | Intervention, randomisation, or valid adjustment |

The first row is the most direct target for a learner trained on $(X,Y)$: the average observed output at observed tilt $x$.

The other rows are stronger claims. They require more than a scatter plot. They may require measuring hidden variables, checking deployment coverage, or using an experimental design.

### Minimal example

The plot below uses one observed training dataset, then compares three different target relationships:

- the observed prediction target estimated from the measured $(X,Y)$ samples;
- the latent physical response $f_0(\theta)$;
- a deployment target where the hidden-context mixture changes.

The editable context modes change $P(C\mid X)$ in the training or deployment process. The same scatter plot can look relevant to all three claims, but it provides direct evidence for only the observed-prediction estimand under the training data source.


In [ ]:
# Change these values, then rerun this cell.
# context mode options: "independent", "increasing", "reversed"
n = 180
seed = 31
context_strength = 0.35
train_context_mode = "increasing"
deployment_context_mode = "reversed"
y_noise = 0.04
bandwidth = 7.0

result = plot_claim_targets_example(
    n=n,
    seed=seed,
    context_strength=context_strength,
    train_context_mode=train_context_mode,
    deployment_context_mode=deployment_context_mode,
    y_noise=y_noise,
    bandwidth=bandwidth,
)


### Plot interpretation

The observed smoother is the most direct target for a learner trained only on $(X,Y)$ from this data source.

The physical curve and the deployment curve are different estimands. They may be close in some regions and different in others, but the observed dataset does not automatically decide between them.

When the printed gaps are large, the same finite dataset is pointing at substantially different answers depending on the claim. That is not a modelling detail. It is an estimand problem.

### Takeaway

Data adequacy is **claim-relative**.

Before fitting a model, decide whether the goal is:

- prediction from observed measurements;
- recovery of a physical mechanism;
- prediction under deployment conditions;
- or a causal/interventional claim.

The same dataset may be adequate for the first claim and inadequate for the others.

The practical question is:

$$
\text{What claim does this dataset actually support?}
$$


<br>

## 3. Where Does the Dataset Provide Evidence?

### Motivation

A fitted curve gives a prediction at every $x$, but the dataset does not provide equal evidence everywhere. The curve may look continuous, while the empirical support is patchy.

For a location $x$ and neighbourhood radius $r$, define the local evidence count:

$$
n_r(x)=\sum_{i=1}^{n}\mathbf{1}\{|x_i-x|\leq r\}.
$$

This count is not a model fit. It is a diagnostic for whether the data contain observations near the location where the fitted curve is being interpreted.

This section asks:

$$
\text{Where is the fitted behaviour constrained by nearby data?}
$$

The distinction matters because predictions in dense regions are mostly evidence-driven, while predictions in gaps or sparse regions depend more heavily on modelling assumptions.

### Minimal example

The plot holds the dataset and fitted smoother fixed. The shaded regions mark locations where $n_r(x)$ falls below the chosen minimum count.

The editable `radius` controls the neighbourhood size $r$, and `min_count` controls how much nearby evidence is treated as enough for this diagnostic.

Look for the difference between:

- regions with many nearby observations;
- regions with few or no nearby observations;
- parts of the fitted curve that are drawn because the smoother must output something.

The key point is:

$$
\text{a model can interpolate across a gap even when the data do not resolve what happens inside it.}
$$


In [ ]:
# Change these values, then rerun this cell.
# sampling options: "uniform", "gap65", "cluster40", "restricted", "sparse_feature", "dense_feature"
n = 105
seed = 45
y_noise = 0.055
sampling = "gap65"
radius = 4.0
min_count = 6
bandwidth = 6.0

result = plot_local_support_example(
    n=n,
    seed=seed,
    y_noise=y_noise,
    sampling=sampling,
    radius=radius,
    min_count=min_count,
    bandwidth=bandwidth,
)


### Plot interpretation

The fitted curve is drawn across the full tilt range, but the evidence is not uniform.

Where observations are dense, nearby data constrain the fitted curve. Where observations are sparse, the fitting rule fills in behaviour using assumptions about smoothness, shape, or model class.

It should be read as:

$$
\text{the curve is more supported where there are relevant nearby observations.}
$$

A claim about behaviour in a weakly supported region, such as a missing local feature near a gap, would require more data in that region or a stronger justified assumption.

The local count $n_r(x)$ is only a support diagnostic. It does not prove the fitted curve is correct. It tells us where the dataset has nearby observations that could constrain the claim.

### Takeaway

The second limit of a dataset is **local support**.

A dataset may support claims in well-covered regions while leaving other regions unresolved. A fitted model will still draw predictions there, but those predictions are partly assumption-driven.

The practical question is:

$$
\text{Where does this dataset provide evidence for the claim we care about?}
$$


<br>

## 4. Does the Dataset Resolve What Matters?

### Motivation

Coverage asks whether observations exist near the region of interest. Resolution asks whether those observations are dense and clean enough to distinguish the behaviour the claim requires.

Let the observed regression function be:

$$
\mu(x)=\mathbb{E}[Y\mid X=x].
$$

A local feature is harder to resolve when it changes quickly, is sampled sparsely, or is hidden by noise. One simple way to describe local behavioural change over radius $r$ is:

$$
\Delta_r(x)=\sup_{u,v\in[x-r,x+r]}|\mu(u)-\mu(v)|.
$$

A rough signal-to-uncertainty diagnostic compares that local change with local sampling variability:

$$
\rho_r(x)=\frac{\Delta_r(x)}{\widehat{\sigma}(x)/\sqrt{n_r(x)}}.
$$

Large $\rho_r(x)$ means the local behaviour is more likely to be distinguishable from noise. Small $\rho_r(x)$ means coverage may exist, but the behaviour may still be unresolved.

This section asks:

$$
\text{Does the dataset resolve the behaviour that matters for the task?}
$$

A dataset does not need to resolve the function equally everywhere. It needs to resolve the parts of the relationship that the research claim or deployment use depends on.

### Minimal example

Use the same underlying curve $f_0$, but change where samples are collected and how noisy they are.

The editable `claim_region` marks the part of $x$-space where the claim matters most, and `min_signal_to_uncertainty` sets the threshold used to flag weak resolution.

Compare cases such as:

- dense sampling near the broad peak;
- sparse sampling near the narrow feature;
- a gap around the narrow feature;
- noisy repeated measurements near the claim region.

Look for whether the data can distinguish a real local feature from noise, smoothing, or model bias.


In [ ]:
# Change these values, then rerun this cell.
# sampling options: "uniform", "sparse_feature", "dense_feature", "gap65", "cluster40"
n = 120
seed = 61
y_noise = 0.08
sampling = "sparse_feature"
radius = 5.0
min_count = 5
min_signal_to_uncertainty = 2.0
claim_region = (58, 72)
bandwidth = 6.0

result = plot_resolution_diagnostic_example(
    n=n,
    seed=seed,
    y_noise=y_noise,
    sampling=sampling,
    radius=radius,
    min_count=min_count,
    min_signal_to_uncertainty=min_signal_to_uncertainty,
    claim_region=claim_region,
    bandwidth=bandwidth,
)


### Plot interpretation

A region can have some coverage but still poor resolution.

If the curve changes slowly, a small number of samples may be enough to estimate broad behaviour. If the curve has a narrow feature, the same number of samples may miss or blur it.

It should be read as:

$$
\text{the data must be dense and clean enough for the claim being made.}
$$

The printed median count and median resolution score summarize this diagnostic inside the claim region. They are not a theorem about learnability. They are a way to ask whether the data are adequate for the local claim.

### Task weighting

Some regions matter more than others. Let $w(x)\geq 0$ describe task importance. A task-weighted risk is:

$$
R_w(h)=\mathbb{E}[w(X)(h(X)-Y)^2].
$$

The question becomes:

$$
\text{Does the dataset resolve the function where }w(x)\text{ is large?}
$$

For example:

- if peak output matters, $w(x)$ is high near the peak;
- if edge cases matter, $w(x)$ is high near the boundaries;
- if local anomalies matter, $w(x)$ is high near narrow features;
- if safety matters, $w(x)$ is high where errors are costly.

### Takeaway

The third limit of a dataset is **resolution**.

A dataset can have samples near a region and still fail to resolve the behaviour needed for the claim. Resolution depends on sample density, noise level, local variation, and task importance.

The practical question is:

$$
\text{Does the dataset resolve what matters for this claim?}
$$


<br>

## 5. What Remains Ambiguous?

### Motivation

Low training error does not mean the data have identified a unique relationship. Several different functions can fit the observed samples well while disagreeing in regions where the data provide little evidence.

To name this formally, define a near-optimal version space:

$$
\mathcal{V}_{\varepsilon}(\mathcal{D})
=
\left\{
h\in\mathcal{H}:
\widehat{R}(h)
\leq
\inf_{h'\in\mathcal{H}}\widehat{R}(h')+\varepsilon
\right\}.
$$

This is the set of functions in $\mathcal{H}$ whose empirical error is close enough to the best available fit. If many functions are in this set, then the data have not selected a unique explanation.

For a claim region $A$, the relevant question is whether those plausible functions agree there:

$$
\operatorname{Dis}_A
=
\sup_{h,h'\in\mathcal{V}_{\varepsilon}(\mathcal{D})}
\sup_{x\in A}|h(x)-h'(x)|.
$$

This section asks:

$$
\text{Which behaviours are ruled out by the data, and which remain possible?}
$$

This matters because a fitted model shows one selected curve, not the full set of explanations still compatible with the observations.

### Minimal example

The plot shows several plausible functions fitted to the same dataset.

The observed points and the data gap are held fixed. The editable `bump_strengths` create candidate functions that agree with the observations but disagree inside the unsupported gap.

Look for the difference between:

- agreement forced by observed data;
- disagreement allowed by missing data;
- the single curve selected by one fitting rule;
- the broader set of curves still compatible with the dataset.

The key point is:

$$
\text{low error on observed samples does not remove ambiguity in unsupported regions.}
$$


In [ ]:
# Change these values, then rerun this cell.
n = 78
seed = 70
y_noise = 0.035
gap_region = (58, 72)
bump_strengths = [-0.13, 0.0, 0.15]

result = plot_ambiguity_example(
    n=n,
    seed=seed,
    y_noise=y_noise,
    gap_region=gap_region,
    bump_strengths=bump_strengths,
)


### Plot interpretation

The curves fit the observed points about equally well. They differ mainly inside the gap, where there are few or no observations.

It should be read as:

$$
\text{the data constrain some behaviours and leave others unresolved.}
$$

If the research claim depends on the narrow feature inside the gap, then the claim is weakly supported. More targeted samples would be needed to distinguish between the plausible alternatives.

The printed maximum candidate disagreement is an empirical version of $\operatorname{Dis}_A$ for the gap region. If that disagreement is large, the data do not resolve the claim in that region.

### Takeaway

The fourth limit of a dataset is **ambiguity**.

A dataset can strongly support behaviour in observed regions while leaving behaviour in unsupported regions unresolved. Low training error only says that a function matches the observed samples. It does not show that all plausible functions agree where the claim matters.

The practical question is:

$$
\text{What explanations remain compatible with the data?}
$$


<br>

## 6. Does the Relationship Survive a Change in Context?

### Motivation

A model can perform well on a random test split and still fail when the data-generating process changes. Random splits usually test performance under the same observation process. They do not automatically test whether the learned relationship is stable across hidden contexts or deployment conditions.

With hidden context, the observed regression function can be decomposed as:

$$
\mathbb{E}[Y\mid X=x]
=
\sum_c \mathbb{E}[Y\mid X=x,C=c]P(C=c\mid X=x).
$$

This equation separates two ingredients:

- the context-specific response, $\mathbb{E}[Y\mid X=x,C=c]$;
- the context mixture, $P(C=c\mid X=x)$.

A random test split usually preserves both ingredients. A deployment shift may preserve the context-specific responses while changing the mixture.

This section asks:

$$
\text{Is the observed relationship stable, or is it tied to the way the data were collected?}
$$

### Minimal example

Generate two hidden contexts:

$$
C\in\{0,1\},
$$

with different response scales:

$$
Y=A_C f_0(\Theta)+\varepsilon_Y.
$$

Then make context depend on tilt. For example, high tilt values may be more likely to appear in one context than the other.

The learner observes only:

$$
X=\Theta,
\qquad
Y.
$$

It does not observe $C$.

The editable context modes change $P(C\mid X)$. Evaluate the same trained model on:

- a random test split with the training mixture;
- a context-balanced test set;
- a shifted test set where $P(C\mid X)$ changes.


In [ ]:
# Change these values, then rerun this cell.
# context mode options: "independent", "increasing", "reversed"
n_train = 180
n_test = 140
seed = 91
context_strength = 0.35
train_context_mode = "increasing"
balanced_context_mode = "independent"
shifted_context_mode = "reversed"
y_noise = 0.05
bandwidth = 7.0

result = plot_context_shift_example(
    n_train=n_train,
    n_test=n_test,
    seed=seed,
    context_strength=context_strength,
    train_context_mode=train_context_mode,
    balanced_context_mode=balanced_context_mode,
    shifted_context_mode=shifted_context_mode,
    y_noise=y_noise,
    bandwidth=bandwidth,
)


### Plot interpretation

The random split may look good because train and test share the same hidden-context mixture.

The shifted test set may fail because the relationship between $X$ and $Y$ has changed. The physical context-specific responses may be unchanged, but the mixture over contexts is different.

So the random split should not be read as:

$$
\text{the model has learned a stable physical response.}
$$

It should be read as:

$$
\text{the model works under the same observed data source.}
$$

If deployment changes the hidden-context mixture, then:

$$
P_{\mathrm{deploy}}(C\mid X)\neq P_{\mathrm{train}}(C\mid X),
$$

and the learned observed relationship may not transfer.

A causal or physical intervention claim is different again. It asks about setting the true tilt:

$$
\mathbb{E}[Y\mid do(\Theta=x)].
$$

That is not the same as merely observing cases with $X=x$.

### Takeaway

The fifth limit of a dataset is **robustness**.

A dataset may support prediction under the observed data source while failing to support deployment or causal claims. The issue is not just model flexibility. The issue is whether the validation design tests the claim being made.

The practical question is:

$$
\text{Will this learned relationship survive the change we actually care about?}
$$


<br>

## 7. Summary and Link to Hypothesis and Optimisation Spaces

The observed dataset is:

$$
\mathcal{D}=\{(x_i,y_i)\}_{i=1}^{n}.
$$

It is only a finite trace of a larger data-generating process:

$$
X=\Theta+\varepsilon_X,
$$

$$
Y=f(\Theta,C)+\varepsilon_Y.
$$

The learner sees $(X,Y)$. The researcher must decide what claim those observations can support.

| Question | Main diagnostic | Research concern |
|---|---|---|
| What can the learner see? | $m_{\mathrm{obs}}(x)=\mathbb{E}[Y\mid X=x]$ | Observability |
| What claim are we making? | Estimand checklist | Estimand clarity |
| Where is there evidence? | $n_r(x)$ | Local support |
| Does the dataset resolve what matters? | $\rho_r(x)$ and $R_w(h)$ | Task-relative adequacy |
| What remains ambiguous? | $\mathcal{V}_{\varepsilon}(\mathcal{D})$ and $\operatorname{Dis}_A$ | Ambiguity |
| Does the relationship survive context shift? | Train/deployment comparison | Robustness |

Core statement:

$$
\text{Data determine what is evidenced, not what is true everywhere.}
$$

Connection to the broader course:

$$
\mathcal{D}\text{ determines what evidence is available.}
$$

$$
\mathcal{H}\text{ determines what functions are possible.}
$$

$$
\mathcal{O}\text{ determines which compatible function is selected.}
$$

The next notebook asks:

$$
\text{Given finite and imperfect data, how does the hypothesis space shape interpolation, extrapolation, and generalisation?}
$$
